# Home Credit Default Risk Scoring

Пет-проект по классическому машинному обучению для кредитного скоринга на датасете **Home Credit Default Risk**.

Цель: предсказать вероятность дефолта клиента (`TARGET = 1`) и сравнить три модели:

- логистическая регрессия из `scikit-learn`;
- random forest из `scikit-learn`;
- градиентный бустинг из `XGBoost`.

Подбор гиперпараметров проводится через `Optuna` по метрике `ROC-AUC` с `timeout = 300` секунд для каждой модели. Итоговое сравнение выполняется по `ROC-AUC`, `PR-AUC` и `Gini`.

## 1. Установка зависимостей

Если зависимости уже установлены, эту ячейку можно пропустить.

In [1]:
%pip install pandas numpy scikit-learn xgboost optuna joblib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Импорты и настройки

Перед запуском положите файл `application_train.csv` из соревнования Kaggle Home Credit Default Risk в папку `data/raw/` рядом с этим ноутбуком.

In [2]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Callable

import joblib
import numpy as np
import optuna
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

RANDOM_STATE = 42
OPTUNA_TIMEOUT = 300
LOGREG_MAX_ITER = 300
LOGREG_TOL = 1e-3
N_TRIALS = None
SAMPLE_ROWS = None

PROJECT_DIR = Path.cwd()
DATA_PATH = PROJECT_DIR / "data" / "raw" / "application_train.csv"
REPORTS_DIR = PROJECT_DIR / "reports"
MODELS_DIR = PROJECT_DIR / "models"

TARGET_COL = "TARGET"
ID_COL = "SK_ID_CURR"
TEST_SIZE = 0.20
VALIDATION_SIZE = 0.20

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

optuna.logging.set_verbosity(optuna.logging.WARNING)

## 3. Загрузка данных

In [3]:
if not DATA_PATH.exists():
    raise FileNotFoundError(
        f"Файл не найден: {DATA_PATH}\n"
        "Скачайте Home Credit Default Risk с Kaggle и положите application_train.csv в data/raw/."
    )

df = pd.read_csv(DATA_PATH)

if SAMPLE_ROWS is not None:
    df = df.sample(n=min(SAMPLE_ROWS, len(df)), random_state=RANDOM_STATE)

print(df.shape)
df.head()

(307511, 122)


,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
3,100006,0,Cash loans,F,N,Y,0,135000.0,312682.5,29686.5,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN
4,100007,0,Cash loans,M,N,Y,0,121500.0,513000.0,21865.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


## 4. Быстрый EDA

Посмотрим на баланс классов и базовую структуру признаков.

In [4]:
display(df[TARGET_COL].value_counts(normalize=True).rename("share"))

summary = pd.DataFrame(
    {
        "dtype": df.dtypes.astype(str),
        "missing_share": df.isna().mean(),
        "n_unique": df.nunique(dropna=True),
    }
).sort_values("missing_share", ascending=False)

summary.head(15)

TARGET
0    0.919271
1    0.080729
Name: share, dtype: float64

,dtype,missing_share,n_unique
COMMONAREA_AVG,float64,0.698723,3181
COMMONAREA_MODE,float64,0.698723,3128
COMMONAREA_MEDI,float64,0.698723,3202
NONLIVINGAPARTMENTS_MEDI,float64,0.694330,214
NONLIVINGAPARTMENTS_MODE,float64,0.694330,167
NONLIVINGAPARTMENTS_AVG,float64,0.694330,386
FONDKAPREMONT_MODE,object,0.683862,4
LIVINGAPARTMENTS_AVG,float64,0.683550,1868
LIVINGAPARTMENTS_MEDI,float64,0.683550,1097
LIVINGAPARTMENTS_MODE,float64,0.683550,736


## 5. Feature engineering

Добавим несколько простых скоринговых признаков: отношения кредита, дохода, аннуитета и стоимости товара. Также заменим техническое значение `365243` в `DAYS_EMPLOYED` на пропуск.

In [5]:
def add_credit_features(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()

    if "DAYS_EMPLOYED" in data.columns:
        data["DAYS_EMPLOYED"] = data["DAYS_EMPLOYED"].replace(365243, np.nan)

    ratio_specs = {
        "CREDIT_INCOME_RATIO": ("AMT_CREDIT", "AMT_INCOME_TOTAL"),
        "ANNUITY_INCOME_RATIO": ("AMT_ANNUITY", "AMT_INCOME_TOTAL"),
        "ANNUITY_CREDIT_RATIO": ("AMT_ANNUITY", "AMT_CREDIT"),
        "GOODS_PRICE_CREDIT_RATIO": ("AMT_GOODS_PRICE", "AMT_CREDIT"),
        "EMPLOYED_BIRTH_RATIO": ("DAYS_EMPLOYED", "DAYS_BIRTH"),
    }

    for new_col, (num_col, den_col) in ratio_specs.items():
        if num_col in data.columns and den_col in data.columns:
            denominator = data[den_col].replace(0, np.nan)
            data[new_col] = data[num_col] / denominator

    feature_cols = [col for col in data.columns if col not in {TARGET_COL, ID_COL}]
    data["MISSING_VALUES_COUNT"] = data[feature_cols].isna().sum(axis=1)

    return data.replace([np.inf, -np.inf], np.nan)


df_model = add_credit_features(df)
X = df_model.drop(columns=[TARGET_COL, ID_COL])
y = df_model[TARGET_COL].astype(int)

print(X.shape, y.shape)

(307511, 126) (307511,)


## 6. Train / validation / test split

Данные делятся на три части:

- train — обучение внутри Optuna;
- validation — выбор лучших гиперпараметров по `ROC-AUC`;
- test — финальное честное сравнение моделей.

In [6]:
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

valid_relative_size = VALIDATION_SIZE / (1.0 - TEST_SIZE)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid,
    y_train_valid,
    test_size=valid_relative_size,
    stratify=y_train_valid,
    random_state=RANDOM_STATE,
)

print("train:", X_train.shape, "valid:", X_valid.shape, "test:", X_test.shape)
print("target rate train:", round(y_train.mean(), 4), "valid:", round(y_valid.mean(), 4), "test:", round(y_test.mean(), 4))

train: (184506, 126) valid: (61502, 126) test: (61503, 126)
target rate train: 0.0807 valid: 0.0807 test: 0.0807


## 7. Preprocessing и модели

In [7]:
def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse=True)


def make_preprocessor(data: pd.DataFrame, scale_numeric: bool) -> ColumnTransformer:
    numeric_features = data.select_dtypes(include=["number", "bool"]).columns.tolist()
    categorical_features = [col for col in data.columns if col not in numeric_features]

    numeric_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        numeric_steps.append(("scaler", StandardScaler(with_mean=False)))

    categorical_pipeline = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", make_one_hot_encoder()),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", Pipeline(steps=numeric_steps), numeric_features),
            ("cat", categorical_pipeline, categorical_features),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )


def make_logistic_regression(trial: optuna.Trial) -> LogisticRegression:
    c_value = trial.suggest_float("C", 1e-3, 3.0, log=True)

    return LogisticRegression(
        C=c_value,
        penalty="l2",
        solver="lbfgs",
        class_weight="balanced",
        max_iter=LOGREG_MAX_ITER,
        tol=LOGREG_TOL,
        random_state=RANDOM_STATE,
    )


def make_random_forest(trial: optuna.Trial) -> RandomForestClassifier:
    return RandomForestClassifier(
        n_estimators=trial.suggest_int("n_estimators", 150, 700, step=50),
        max_depth=trial.suggest_int("max_depth", 4, 18),
        min_samples_split=trial.suggest_int("min_samples_split", 2, 60),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 1, 40),
        max_features=trial.suggest_categorical("max_features", ["sqrt", "log2", 0.4, 0.7]),
        class_weight=trial.suggest_categorical("class_weight", ["balanced", "balanced_subsample"]),
        bootstrap=True,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )


def class_imbalance_weight(target: pd.Series) -> float:
    positives = np.sum(target == 1)
    negatives = np.sum(target == 0)
    return float(negatives / positives) if positives > 0 else 1.0


def make_xgboost(trial: optuna.Trial) -> XGBClassifier:
    return XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators", 200, 900, step=50),
        max_depth=trial.suggest_int("max_depth", 2, 8),
        learning_rate=trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        subsample=trial.suggest_float("subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
        min_child_weight=trial.suggest_float("min_child_weight", 1.0, 20.0),
        gamma=trial.suggest_float("gamma", 0.0, 10.0),
        reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("reg_lambda", 1e-3, 30.0, log=True),
        objective="binary:logistic",
        eval_metric="auc",
        tree_method="hist",
        scale_pos_weight=class_imbalance_weight(y_train),
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )

## 8. Optuna tuning по ROC-AUC

In [8]:
def tune_model(
    model_name: str,
    estimator_factory: Callable[[optuna.Trial], object],
    preprocessor: ColumnTransformer,
    timeout: int = OPTUNA_TIMEOUT,
    n_trials: int | None = N_TRIALS,
) -> tuple[Pipeline, optuna.Study]:
    print(f"Preparing features for {model_name}...")
    tuning_preprocessor = clone(preprocessor)
    X_train_tuned = tuning_preprocessor.fit_transform(X_train)
    X_valid_tuned = tuning_preprocessor.transform(X_valid)
    print(f"Prepared {model_name} matrices: train={X_train_tuned.shape}, valid={X_valid_tuned.shape}")

    def objective(trial: optuna.Trial) -> float:
        estimator = estimator_factory(trial)
        estimator.fit(X_train_tuned, y_train)
        valid_proba = estimator.predict_proba(X_valid_tuned)[:, 1]
        return float(roc_auc_score(y_valid, valid_proba))

    sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction="maximize", sampler=sampler, study_name=model_name)
    study.optimize(objective, timeout=timeout, n_trials=n_trials, show_progress_bar=True)

    best_pipeline = Pipeline(
        steps=[
            ("preprocess", clone(preprocessor)),
            ("model", estimator_factory(study.best_trial)),
        ]
    )
    return best_pipeline, study


def evaluate_model(
    model_name: str,
    pipeline: Pipeline,
    study: optuna.Study,
) -> dict[str, object]:
    print(f"Final fit for {model_name} on train+validation...")
    pipeline.fit(X_train_valid, y_train_valid)
    print(f"Predicting test probabilities for {model_name}...")
    test_proba = pipeline.predict_proba(X_test)[:, 1]

    roc_auc = float(roc_auc_score(y_test, test_proba))
    pr_auc = float(average_precision_score(y_test, test_proba))
    gini = 2.0 * roc_auc - 1.0

    return {
        "model": model_name,
        "valid_roc_auc": float(study.best_value),
        "test_roc_auc": roc_auc,
        "test_pr_auc": pr_auc,
        "test_gini": gini,
        "n_trials": len(study.trials),
        "best_params": study.best_params,
    }

## 9. Обучение и сравнение моделей

Эта ячейка запускает Optuna-подбор для каждой модели. При `OPTUNA_TIMEOUT = 300` весь блок будет выполняться до 300 секунд как в случае логистической регрессии, так и в случае random forest и  XGBoost.

In [9]:
model_specs = {
    "logistic_regression": {
        "preprocessor": make_preprocessor(X_train, scale_numeric=True),
        "factory": make_logistic_regression,
    },
    "random_forest": {
        "preprocessor": make_preprocessor(X_train, scale_numeric=False),
        "factory": make_random_forest,
    },
    "xgboost": {
        "preprocessor": make_preprocessor(X_train, scale_numeric=False),
        "factory": make_xgboost,
    },
}

results = []
studies_summary = {}

for model_name, spec in model_specs.items():
    print(f"\n=== Tuning {model_name} with Optuna ROC-AUC objective ===")
    pipeline, study = tune_model(
        model_name=model_name,
        estimator_factory=spec["factory"],
        preprocessor=spec["preprocessor"],
        timeout=OPTUNA_TIMEOUT,
        n_trials=N_TRIALS,
    )

    result = evaluate_model(model_name, pipeline, study)
    results.append(result)

    studies_summary[model_name] = {
        "best_valid_roc_auc": float(study.best_value),
        "best_params": study.best_params,
        "n_trials": len(study.trials),
    }

    print(f"Saving {model_name} pipeline...")
    joblib.dump(pipeline, MODELS_DIR / f"{model_name}.joblib")
    print(f"Finished {model_name}")

results_df = pd.DataFrame(results).sort_values("test_roc_auc", ascending=False)
results_df_display = results_df.drop(columns=["best_params"])

results_df.to_csv(REPORTS_DIR / "model_comparison.csv", index=False)
with open(REPORTS_DIR / "optuna_best_params.json", "w", encoding="utf-8") as file:
    json.dump(studies_summary, file, indent=2, ensure_ascii=False)

display(results_df_display)


=== Tuning logistic_regression with Optuna ROC-AUC objective ===
Preparing features for logistic_regression...
Prepared logistic_regression matrices: train=(184506, 247), valid=(61502, 247)


   0%|          | 00:00/05:00

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stab

Final fit for logistic_regression on train+validation...


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


...

   0%|          | 00:00/05:00

Final fit for random_forest on train+validation...
Predicting test probabilities for random_forest...
Saving random_forest pipeline...
Finished random_forest

=== Tuning xgboost with Optuna ROC-AUC objective ===
Preparing features for xgboost...
Prepared xgboost matrices: train=(184506, 247), valid=(61502, 247)


   0%|          | 00:00/05:00

Final fit for xgboost on train+validation...
Predicting test probabilities for xgboost...
Saving xgboost pipeline...
Finished xgboost


,model,valid_roc_auc,test_roc_auc,test_pr_auc,test_gini,n_trials
2,xgboost,0.764492,0.770475,0.262807,0.540950,23
0,logistic_regression,0.746126,0.750893,0.231157,0.501786,14
1,random_forest,0.742506,0.749707,0.234809,0.499415,1


## 10. Краткая сравнительная характеристика

- `ROC-AUC` оценивает качество ранжирования: насколько часто дефолтный клиент получает более высокий риск, чем недефолтный.
- `PR-AUC` особенно важна при дисбалансе классов, потому что дефолтов обычно намного меньше.
- `Gini` часто используется в кредитном скоринге и равен `2 * ROC-AUC - 1`.

Обычно в этой задаче XGBoost должен быть сильным кандидатом за счет нелинейностей и взаимодействий признаков. Логистическая регрессия полезна как интерпретируемый baseline, а random forest часто дает устойчивый, но не всегда лучший результат на табличных скоринговых данных.

In [10]:
best_model_row = results_df.iloc[0]
print("Лучшая модель по test ROC-AUC:", best_model_row["model"])
print("Test ROC-AUC:", round(best_model_row["test_roc_auc"], 5))
print("Test PR-AUC:", round(best_model_row["test_pr_auc"], 5))
print("Test Gini:", round(best_model_row["test_gini"], 5))

results_df[["model", "valid_roc_auc", "test_roc_auc", "test_pr_auc", "test_gini", "n_trials"]]

Лучшая модель по test ROC-AUC: xgboost
Test ROC-AUC: 0.77047
Test PR-AUC: 0.26281
Test Gini: 0.54095


,model,valid_roc_auc,test_roc_auc,test_pr_auc,test_gini,n_trials
2,xgboost,0.764492,0.770475,0.262807,0.540950,23
0,logistic_regression,0.746126,0.750893,0.231157,0.501786,14
1,random_forest,0.742506,0.749707,0.234809,0.499415,1
